In [1]:
import pandas as pd

import re

In [3]:
df = pd.read_parquet('..\\data\\raw\\neo4j-2024v1\\train-00000-of-00001.parquet')
df

,question,schema,cypher,data_source,instance_id,database_reference_alias
0,Which 3 countries have the most entities linke...,Node properties:\n- **Country**\n - `location...,MATCH (f:Filing)-[:BENEFITS]->(e:Entity)-[:COU...,neo4jLabs_synthetic_gpt4o,instance_id_41185,neo4jlabs_demo_db_fincen
1,What are the names of the first 3 organization...,Node properties:\n- **Person**\n - `name`: ST...,MATCH (o:Organization)-[:HAS_CEO]->(ceo:Person...,neo4jLabs_synthetic_gpt4turbo,instance_id_26598,neo4jlabs_demo_db_companies
2,List the names of the games played by streams ...,Node properties:\n- **Stream**\n - `createdAt...,MATCH (s:Stream)-[:PLAYS]->(g:Game) WHERE s.to...,neo4jLabs_synthetic_gemini,instance_id_34035,neo4jlabs_demo_db_twitch
3,For each Article find its abstract and the cou...,Graph schema: Relevant node labels and their p...,MATCH (n:Article) -[:HAS_KEY]->(m:Keyword) WIT...,neo4jLabs_functional_cypher,instance_id_3914,None
4,Find the Author for which first_name starts wi...,Graph schema: Relevant node labels and their p...,MATCH (n:Author) WHERE n.first_name STARTS WIT...,neo4jLabs_functional_cypher,instance_id_14645,None
...,...,...,...,...,...,...
39549,Who are the top 5 users with x-coordinate valu...,Node properties:\n- **User**\n - `label`: STR...,MATCH (u:User) WHERE u.x < -5000 RETURN u.labe...,neo4jLabs_synthetic_gpt4o,instance_id_40775,neo4jlabs_demo_db_bluesky
39550,Find the shortest path between Journal where j...,Graph schema: Relevant node labels and their p...,MATCH p=shortestPath((a:Journal{journal_id:'f7...,neo4jLabs_functional_cypher,instance_id_6588,None
39551,Return the first_name for Author combined with...,Graph schema: Relevant node labels and their p...,MATCH (n:Author) RETURN n.first_name AS Record...,neo4jLabs_functional_cypher,instance_id_16140,None
39552,Which 3 users have the highest rate of answere...,Node properties:\n- **Question**\n - `favorit...,MATCH (u:User)-[:ASKED]->(q:Question) WHERE q....,neo4jLabs_synthetic_gpt4o,instance_id_40363,neo4jlabs_demo_db_buzzoverflow


In [4]:
# count and shows unique database_reference_alias column values
df['database_reference_alias'].value_counts(dropna=False)

database_reference_alias
None                                 17461
neo4jlabs_demo_db_eoflix              2512
neo4jlabs_demo_db_companies           2422
neo4jlabs_demo_db_recommendations     2130
neo4jlabs_demo_db_movies              1949
neo4jlabs_demo_db_northwind           1669
neo4jlabs_demo_db_twitch              1604
neo4jlabs_demo_db_grandstack          1492
neo4jlabs_demo_db_fincen              1347
neo4jlabs_demo_db_gameofthrones       1296
neo4jlabs_demo_db_twitter             1284
neo4jlabs_demo_db_buzzoverflow        1106
neo4jlabs_demo_db_network             1094
neo4jlabs_demo_db_offshoreleaks        921
neo4jlabs_demo_db_stackoverflow2       867
neo4jlabs_demo_db_bluesky              388
neo4jlabs_demo_db_openstreetmap         10
neo4jlabs_demo_db_stackoverflow          2
Name: count, dtype: int64

In [5]:
with_db_mask = df['database_reference_alias'].notnull()
# print how many rows have database_reference_alias
print(f"Rows with database_reference_alias: {with_db_mask.sum()}")

Rows with database_reference_alias: 22093


In [6]:
# filter rows that do not have database_reference_alias
to_remove = df[~with_db_mask]
df = df[with_db_mask]
print(f"Removing {len(to_remove)} rows that do not have database_reference_alias")
print(f"Remaining rows: {len(df)}")
print("Removed rows:")
to_remove

Removing 17461 rows that do not have database_reference_alias
Remaining rows: 22093
Removed rows:


,question,schema,cypher,data_source,instance_id,database_reference_alias
3,For each Article find its abstract and the cou...,Graph schema: Relevant node labels and their p...,MATCH (n:Article) -[:HAS_KEY]->(m:Keyword) WIT...,neo4jLabs_functional_cypher,instance_id_3914,None
4,Find the Author for which first_name starts wi...,Graph schema: Relevant node labels and their p...,MATCH (n:Author) WHERE n.first_name STARTS WIT...,neo4jLabs_functional_cypher,instance_id_14645,None
6,List nodes that are 3 hops away from Article f...,Graph schema: Relevant node labels and their p...,MATCH (a:Article{title:'Maslov class and minim...,neo4jLabs_functional_cypher,instance_id_17972,None
8,Search for the title values from 20 Article th...,Graph schema: Relevant node labels and their p...,MATCH (n:Article) -[:HAS_KEY]->(m:Keyword) WIT...,neo4jLabs_functional_cypher,instance_id_3825,None
10,Find the shortest path between Categories wher...,Graph schema: Relevant node labels and their p...,MATCH p=shortestPath((a:Categories{category_id...,neo4jLabs_functional_cypher,instance_id_6440,None
...,...,...,...,...,...,...
39543,Retrieve the Author where affiliation or last_...,Graph schema: Relevant node labels and their p...,MATCH (n:Author) WHERE n.affiliation CONTAINS ...,neo4jLabs_functional_cypher,instance_id_18761,None
39544,Is there a path connecting Journal where journ...,Graph schema: Relevant node labels and their p...,MATCH (a:Journal{journal_id:'f762cb2c3b5bd7f0b...,neo4jLabs_functional_cypher,instance_id_4120,None
39546,Find all offences where a 'weapon' type object...,"Node properties are the following: "":Person {s...",MATCH (c:Crime)-[:INVOLVED_IN]->(o:Object {typ...,hf_vedana17_train,instance_id_2828,None
39550,Find the shortest path between Journal where j...,Graph schema: Relevant node labels and their p...,MATCH p=shortestPath((a:Journal{journal_id:'f7...,neo4jLabs_functional_cypher,instance_id_6588,None
